# Calculate Trip Bundling Metrics for all users >=2 POI stops
Author: Callie Clark

In [1]:
!pip install pydeck -q -q
exec(open('../EV00_settings.py').read())
from shapely import wkt
from scipy.spatial import cKDTree

In [2]:
import numpy as np
import pandas as pd
def prepare_poi_visits(df_iter, user_info_iter):
    #print('user_info_iter', user_info_iter.columns)
    # merge user attributes onto visits
    df = df_iter.merge(
        user_info_iter[['cuebiq_id','tot_pings','date']],
        on='cuebiq_id',
        how='inner'
    )

    # define groups

    df['group'] = np.select(
        [
            df['ev_driver'] == 0,
            (df['ev_driver'] == 1) & (df['charge_day'] == 1),
            (df['ev_driver'] == 1) & (df['charge_day'] == 0)
        ],
        [
            "non_ev",
            "ev_charge",
            "ev_noncharge"
        ],
        default=np.nan
    )

    df = df.dropna(subset=['group'])

    return df

import numpy as np

def calculate_radius_of_gyration(group):

    latitudes = group['lat'].values
    longitudes = group['lng'].values
    weights = group['dwell_time_minutes'].values

    if len(latitudes) < 2:
        return np.nan

    # weighted center of mass
    center_lat = np.average(latitudes, weights=weights)
    center_lon = np.average(longitudes, weights=weights)

    # distances from center
    distances = haversine_vectorized(
        center_lat,
        center_lon,
        latitudes,
        longitudes
    )

    # radius of gyration
    rg = np.sqrt(np.sum(weights * distances**2) / np.sum(weights))

    return rg

def compute_user_day_metrics(df):

    df = df.copy()

    # ensure datetime
    df['stop_zoned_datetime'] = pd.to_datetime(df['stop_zoned_datetime'], errors='coerce')
    #print(df.columns)
    # keep only user-days with >=2 POI stops
    counts = df.groupby(['cuebiq_id','date']).size()
    valid = counts[counts >= 2].index

    df = df.set_index(['cuebiq_id','date']).loc[valid].reset_index()

    # sort stops chronologically
    df = df.sort_values(['cuebiq_id','date','stop_zoned_datetime'])

    # next stop info
    df['next_time'] = df.groupby(['cuebiq_id','date'])['stop_zoned_datetime'].shift(-1)
    df['next_lat'] = df.groupby(['cuebiq_id','date'])['lat'].shift(-1)
    df['next_lon'] = df.groupby(['cuebiq_id','date'])['lng'].shift(-1)

    # time between stops (minutes)
    df['time_between'] = ((
        df['next_time'] - df['stop_zoned_datetime']
    ).dt.total_seconds() / 60)-df['dwell_time_minutes']

    # distance between stops (meters)
    mask = df['next_lat'].notna()

    # aggregate per user per day

    user_day_metrics = df.groupby(['cuebiq_id','date']).agg(
        avg_stop_duration=('dwell_time_minutes','mean'),
        avg_time_between=('time_between','mean'),
        #avg_distance_between=('distance_between','mean'),
        num_stops=('stop_zoned_datetime','count')
    ).reset_index()
    
        # calculate radius of gyration
    rg = (
        df.groupby(['cuebiq_id','date'])
        .apply(calculate_radius_of_gyration)
        .rename('radius_of_gyration')
        .reset_index()
    )

    # merge
    user_day_metrics = user_day_metrics.merge(
        rg,
        on=['cuebiq_id','date'],
        how='left'
    )

    return user_day_metrics

def add_groups(user_day_metrics, user_info):

    df = user_day_metrics.merge(
        user_info[['cuebiq_id','date','ev_driver','charge_day']],
        on=['cuebiq_id','date'],
        how='left'
    )

    df['group'] = np.select(
        [
            df['ev_driver'] == 0,
            (df['ev_driver'] == 1) & (df['charge_day'] == 1),
            (df['ev_driver'] == 1) & (df['charge_day'] == 0)
        ],
        [
            "non_ev",
            "ev_charge",
            "ev_noncharge"
        ]
    )

    return df

def compute_metric_stats(df):

    metrics = [
        'avg_stop_duration',
        'avg_time_between',
        'radius_of_gyration'
    ]

    results = {}

    for g, gdf in df.groupby('group'):

        results[g] = {}

        for m in metrics:

            vals = gdf[m].dropna()

            n = len(vals)
            mean = vals.mean()
            std = vals.std()

            ci = 1.96 * std / np.sqrt(n)

            results[g][m] = {
                "mean": mean,
                "ci_low": mean-ci,
                "ci_high": mean+ci,
                "n": n
            }

    return results



In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime as dt, timedelta

city_metric_results = {}

for current_city in ['SF', 'Denver', 'Boston','Seattle']:#

    start_dt = dt.strptime(start_date, '%Y%m%d')
    end_dt = dt.strptime(end_date, '%Y%m%d')

    current_date = start_dt

    user_info = pd.read_csv(f'{current_city}_daily_user_info.csv', index_col=0)
    user_info.date = user_info.date.astype(str)
    user_info.cuebiq_id = user_info.cuebiq_id.astype(str)

    user_day_list = []

    while current_date <= end_dt:

        date_str = current_date.strftime('%Y%m%d')
        print(current_city, date_str)

        # filter valid users
        user_info_iter = user_info[
            (user_info['date'] == date_str) &
            (user_info['poi_stops'] >= 2) &
            (user_info['tot_stops'] > 2) &
            (user_info['tot_pings'] >= 36) &
            (user_info['tot_pings'] < 2500)
        ].copy()

        if len(user_info_iter) == 0:
            current_date += timedelta(days=1)
            continue

        # load POI stops
        df_iter = pd.read_csv(f'{current_city}_poi_stop_counts_{date_str}.csv') #this is just POI stops 
        df_iter.cuebiq_id = df_iter.cuebiq_id.astype(str)
        df_iter=df_iter[df_iter['category']!='no poi'].copy()
        df_iter=df_iter[df_iter['category']!='Other'].copy()
        

        # filter + assign groups
        df = prepare_poi_visits(df_iter, user_info_iter)

        # compute stop metrics
        user_day_metrics = compute_user_day_metrics(df)

        # add EV groups
        user_day_metrics = add_groups(user_day_metrics, user_info_iter)

        user_day_list.append(user_day_metrics)

        current_date += timedelta(days=1)

    # combine entire timeframe
    all_user_days = pd.concat(user_day_list, ignore_index=True)

    # compute final statistics across all user-days
    metric_stats = compute_metric_stats(all_user_days)

    city_metric_results[current_city] = metric_stats

    # save user-day dataset
    all_user_days.to_csv(f'{current_city}_user_day_mobility_metrics.csv', index=False)

In [5]:
summary_rows = []

for city, stats in city_metric_results.items():

    for group in stats:

        for metric in stats[group]:

            s = stats[group][metric]

            summary_rows.append({
                "city": city,
                "group": group,
                "metric": metric,
                "mean": s["mean"],
                "ci_low": s["ci_low"],
                "ci_high": s["ci_high"],
                "n": s["n"]
            })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv("city_mobility_metric_summary.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_metric_summary(summary_df, fontsize=18):

    metrics = [
        "avg_stop_duration",
        "avg_time_between",
        "radius_of_gyration"
    ]


    cities = ["SF","Seattle", "Boston", "Denver"]
    groups = ["ev_charge", "ev_noncharge", "non_ev"]

    group_labels = {
        "ev_charge": "EV Drivers (Charging Day)",
        "ev_noncharge": "EV Drivers (Non-Charging Day)",
        "non_ev": "Non-EV Drivers"
    }

    colors = {
        "ev_charge": "#6BAF92",      # soft green
        "ev_noncharge": "#6C8EBF",   # muted blue
        "non_ev": "#9A8F97"          # neutral gray-purple
    }

    metric_titles = {
        "avg_stop_duration": "Average Stop Duration",
        "avg_time_between": "Average Time Between Stops",
        "radius_of_gyration": "Daily Radius of Gyration "
    }

    metric_units = {
        "avg_stop_duration": "Min.",
        "avg_time_between": "Min.",
        "radius_of_gyration": "KM"
    }

    n_groups = len(groups)
    x = np.arange(len(cities))
    width = 0.18

    #fig, axes = plt.subplots(1, len(metrics), figsize=(18,6))
    fig, axes = plt.subplots(1, len(metrics), figsize=(12,4.5))
    fig.subplots_adjust(wspace=0.25)

    for ax, metric in zip(axes, metrics):

        df_m = summary_df[summary_df['metric'] == metric]

        for i, group in enumerate(groups):

            df_g = df_m[df_m['group'] == group].set_index('city').reindex(cities)

            means = df_g['mean'].values
            ci_low = df_g['ci_low'].values
            ci_high = df_g['ci_high'].values
            
            if metric == "radius_of_gyration":
                means = means / 1000
                ci_low = ci_low / 1000
                ci_high = ci_high / 1000

            yerr = np.vstack([means - ci_low, ci_high - means])

            ax.bar(
                x + (i - n_groups/2) * width + width/2,
                means,
                width,
                yerr=yerr,
                capsize=5,
                color=colors[group],
                label=group_labels[group],
                alpha=0.95
            )

        ax.set_xticks(x)
        ax.set_xticklabels(cities, fontsize=fontsize)

        ax.set_title(metric_titles[metric], fontsize=fontsize+2)

        ax.set_ylabel(metric_units[metric], fontsize=fontsize)

        ax.tick_params(axis='y', labelsize=fontsize)
        ax.set_xticklabels(cities, rotation=20)

    # Legend across top
    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=fontsize,
        bbox_to_anchor=(0.5, 0.92)
    )

    fig.suptitle(
        "User Behavior during ≥ 2 Daily POI Stops",
        fontsize=fontsize+4,
       y=0.96
    )
    
    plt.tight_layout(rect=[0,0,1,0.88])

    plt.show()